# 实践项目 08：dMRI 与 IHC 图像分析

本参考版本使用 ODBB 官方 `Human Callosum MRI-PLI-Histology` 预览对象。运行目标是用 MRI 的 FA、MD 和片内坐标估计同一教学网格上的 PLI Retardance 结构图。GFAP IHC 图片是独立的外观示例，不是配对模型目标；论文 patch、差异图和棕色面积代理没有进入本 Notebook 的活动数据。

## 任务

1. 核对官方数据对象和数组。
2. 运行 StandardScaler + Ridge 基线。
3. 检查空间块留出。
4. 保存结果和证据边界。


In [ ]:
from pathlib import Path  # 管理跨本地电脑与 Kaggle 的路径
import json  # 读取来源清单并保存结果
import numpy as np  # 处理 MRI/PLI 数组
import matplotlib.pyplot as plt  # 绘制数据和预测图
from scipy.ndimage import gaussian_filter  # 对基线预测做轻度平滑
from sklearn.linear_model import Ridge  # 使用可解释的岭回归
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # 计算评价指标
from sklearn.pipeline import make_pipeline  # 连接标准化与回归
from sklearn.preprocessing import StandardScaler  # 只在训练像素上拟合标准化


def local_roots():  # 兼容从仓库、材料包及其子目录直接运行
    current = Path.cwd().resolve()
    return [current, *current.parents]


def find_data_root():  # 在 Kaggle、本地仓库和学生材料中寻找同一数据包
    candidates = [
        Path("/kaggle/input/kydw-experience-08-data"),
        Path("/kaggle/input/kydw-advanced-08-data"),
    ]
    for root in local_roots():  # 从当前目录逐级向上兼容不同启动位置
        candidates.extend([
            root / "experience" / "data" / "project-09",
            root / "数据",
            root / "data",
            root / "dataset",
            root / "advanced" / "dataset",
        ])
    for candidate in candidates:  # 逐个检查数据包是否存在
        if (candidate / "odbb_simplified.npz").exists():  # 用核心数组作为数据包标志
            return candidate
    raise FileNotFoundError("没有找到 ODBB 派生包 odbb_simplified.npz")  # 给出明确的路径错误


DATA_ROOT = find_data_root()  # 固定本次运行的数据根目录
WORK_ROOT = Path("/kaggle/working")  # Kaggle 的可写目录
if not WORK_ROOT.exists():  # 本地运行时使用仓库临时目录
    WORK_ROOT = Path.cwd() / "tmp"  # 保留输出，便于回看
OUT = WORK_ROOT / "project08_experience_outputs"  # 保存本次参考结果
OUT.mkdir(parents=True, exist_ok=True)  # 创建输出目录
z = np.load(DATA_ROOT / "odbb_simplified.npz", allow_pickle=True)  # 读取官方数据的轻量派生包
manifest = json.loads((DATA_ROOT / "source_manifest.json").read_text(encoding="utf-8"))  # 读取数据来源清单
print({"data_root": str(DATA_ROOT), "dataset": manifest["dataset"]["name"], "id": manifest["dataset"]["id"]})  # 显示来源身份


## 任务 1：官方数据核对

先确认数组和来源清单，再绘制 FA、MD、PLI Retardance。

In [ ]:
sample_id = z["sample_id"].astype(str)  # 读取官方预览对象编号
fa = z["mri_fa"].astype(float)  # 读取 MRI fractional anisotropy
md = z["mri_md"].astype(float)  # 读取 MRI mean diffusivity
retardance = z["pli_retardance"].astype(float)  # 读取 PLI Retardance 结构目标
mri_mask = z["mri_mask"].astype(bool)  # 读取 MRI 有效区域
data_summary = {"sample_id": sample_id.tolist(), "fa_shape": list(fa.shape), "md_shape": list(md.shape), "retardance_shape": list(retardance.shape), "mask_fraction": float(mri_mask.mean()), "fa_range": [float(fa.min()), float(fa.max())], "retardance_range": [float(retardance.min()), float(retardance.max())]}  # 汇总数组契约
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), facecolor="white")  # 建立三列官方数据图
for ax, image, title in zip(axes, [fa, md, retardance], ["MRI · FA", "MRI · MD", "PLI · Retardance"]):  # 逐列绘制同一教学网格
    ax.imshow(np.ma.masked_where(~mri_mask, image), cmap="gray", vmin=0, vmax=1)  # 只在 MRI 有效区显示数值
    ax.set_title(title, loc="left", fontweight="bold", color="#173b51")  # 标注模态和字段
    ax.axis("off")  # 隐藏坐标轴以突出空间结构
fig.tight_layout()  # 收紧图像边距
fig.savefig(OUT / "task1_official_data.png", dpi=170, bbox_inches="tight")  # 保存官方数据概览
plt.show()  # 在 Notebook 中显示图像
print(data_summary)  # 输出可以核对的数组信息


## 任务 2：透明的 MRI→PLI 基线

标准化 FA、MD 和归一化坐标，拟合 Ridge，并观察预测与绝对误差。

In [ ]:
x_coordinate = z["x_coordinate"].astype(float)  # 读取归一化横坐标
y_coordinate = z["y_coordinate"].astype(float)  # 读取归一化纵坐标
features = np.stack([fa, md, x_coordinate, y_coordinate], axis=-1)  # 把两个 MRI 特征和片内布局组成四通道输入
valid = mri_mask & (retardance > 0.04)  # 排除视野外空白和极低目标值
train_mask = z["train_mask"].astype(bool) & valid  # 使用预先提供的训练空间块
model = make_pipeline(StandardScaler(), Ridge(alpha=3.0))  # 建立标准化加岭回归的透明基线
model.fit(features[train_mask], retardance[train_mask])  # 只用训练像素拟合模型
raw_prediction = model.predict(features.reshape(-1, features.shape[-1])).reshape(retardance.shape)  # 对整个教学网格预测
prediction = gaussian_filter(np.clip(raw_prediction, 0, 1), sigma=1.0)  # 轻度平滑，减少单像素噪声
prediction[~mri_mask] = 0  # 保持 MRI 视野外为背景
absolute_error = np.abs(prediction - retardance)  # 计算每个网格位置的绝对误差
fig, axes = plt.subplots(2, 2, figsize=(9, 7.2), facecolor="white")  # 建立输入、目标、输出和误差图
panels = [(fa, "Input · FA", "gray"), (md, "Input · MD", "gray"), (retardance, "Target · PLI Retardance", "gray"), (prediction, "Ridge prediction", "viridis")]  # 准备四个视觉对象
for ax, (image, title, cmap) in zip(axes.flat, panels):  # 逐个展示模型输入和结果
    ax.imshow(np.ma.masked_where(~mri_mask, image), cmap=cmap, vmin=0, vmax=1)  # 使用相同有效区域和数值范围
    ax.set_title(title, loc="left", fontweight="bold", color="#173b51")  # 标出每张图的角色
    ax.axis("off")  # 隐藏坐标轴
axes[1, 1].imshow(np.ma.masked_where(~mri_mask, absolute_error), cmap="magma", vmin=0, vmax=0.7)  # 在输出图上覆盖误差
axes[1, 1].set_title("Absolute error", loc="left", fontweight="bold", color="#173b51")  # 更新误差图标题
fig.tight_layout()  # 调整图间距
fig.savefig(OUT / "task2_ridge_prediction.png", dpi=170, bbox_inches="tight")  # 保存基线结果图
plt.show()  # 显示预测结果
print({"features": ["FA", "MD", "x_coordinate", "y_coordinate"], "train_pixels": int(train_mask.sum()), "alpha": 3.0, "prediction_range": [float(prediction.min()), float(prediction.max())]})  # 输出模型设置


## 任务 3：空间留出评价

训练块与留出块使用同一指标，空间留出不参与模型拟合。

In [ ]:
test_mask = z["test_mask"].astype(bool) & valid  # 读取未参与拟合的空间留出块
def score(mask):  # 在指定空间区域计算同一组指标
    y_true = retardance[mask]  # 取出目标值
    y_pred = prediction[mask]  # 取出预测值
    mse = mean_squared_error(y_true, y_pred)  # 计算均方误差
    return {"pixels": int(mask.sum()), "mae": float(mean_absolute_error(y_true, y_pred)), "rmse": float(np.sqrt(mse)), "r2": float(r2_score(y_true, y_pred)), "correlation": float(np.corrcoef(y_true, y_pred)[0, 1])}  # 返回可保存的数值
training_metrics = score(train_mask)  # 评价训练空间块
holdout_metrics = score(test_mask)  # 评价空间留出块
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.8), facecolor="white")  # 建立目标和留出标记图
axes[0].imshow(np.ma.masked_where(~mri_mask, retardance), cmap="gray", vmin=0, vmax=1)  # 显示官方 PLI 目标
axes[0].set_title("Target · PLI Retardance", loc="left", fontweight="bold", color="#173b51")  # 标注目标
axes[1].imshow(np.ma.masked_where(~mri_mask, fa), cmap="gray", vmin=0, vmax=1)  # 使用 FA 作为留出底图
axes[1].imshow(np.ma.masked_where(~test_mask, test_mask), cmap="Oranges", alpha=0.8)  # 覆盖未参与拟合的空间块
axes[1].set_title("Spatial holdout", loc="left", fontweight="bold", color="#173b51")  # 标注划分
for ax in axes:  # 统一关闭坐标轴
    ax.axis("off")  # 只保留空间位置
fig.tight_layout()  # 调整边距
fig.savefig(OUT / "task3_spatial_holdout.png", dpi=170, bbox_inches="tight")  # 保存空间划分图
plt.show()  # 显示空间留出
metrics = {"training_blocks": training_metrics, "spatial_holdout": holdout_metrics}  # 汇总训练与留出指标
print(metrics)  # 输出可与数据清单核对的指标


## 任务 4：证据边界

将官方数据、独立 IHC 示例和模型边界保存为一个摘要。

In [ ]:
result_summary = {  # 整理数据、模型和解释边界
    "dataset": manifest["dataset"],  # 保存官方数据身份
    "derived_package": manifest["derived_package"],  # 保存缩放和粗对应关系
    "model": {"name": "StandardScaler + Ridge(alpha=3.0) + Gaussian smoothing(sigma=1.0)", "features": ["FA", "MD", "normalized x coordinate", "normalized y coordinate"], "target": "official ODBB PLI Retardance on a teaching grid", "metrics": metrics},  # 保存模型设置和指标
    "ihc_example": manifest["independent_ihc_example"],  # 标记 IHC 图片为独立外观示例
    "boundary": "This is a one-preview-specimen, coarse MRI-to-PLI structural proxy. It is not paired IHC supervision, cell-level protein prediction, or clinical validation.",  # 写明证据边界
}  # 完成结果摘要
(OUT / "result_summary.json").write_text(json.dumps(result_summary, ensure_ascii=False, indent=2), encoding="utf-8")  # 保存结果 JSON
print(result_summary)  # 输出最终摘要
